In [1]:
import pyterrier as pt
import pandas as pd
import numpy as np
import xgboost as xgb

c:\Users\ne3na\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if not pt.started():
        pt.init()



dataset = pt.get_dataset("vaswani")
topics = dataset.get_topics()
qrels = dataset.get_qrels()
print(len(topics), len(qrels))

PyTerrier 0.10.1 has loaded Terrier 5.11 (built by craig.macdonald on 2025-01-13 21:29) and terrier-helper 0.0.8



93 2083


In [18]:
# docs = []
# for item in dataset.get_corpus_iter(verbose=True):
#     doc_id = item.get("docno")
#     doc_text = item.get("text")
#     docs.append((doc_id, doc_text))
#     # count += 1
#     # if count == 500:
#     #     break
# # lenth of the documents
# print("Loaded from data:",len(docs))
topics = dataset.get_topics()
qrels = dataset.get_qrels()
print(len(topics), len(qrels))

93 2083


In [3]:
index_ref = dataset.get_index()
bm25 = pt.BatchRetrieve(index_ref, wmodel="BM25", num_results=1000)
retrieval_df = bm25.transform(topics)

data.direct.bf: 100%|██████████| 388k/388k [00:00<00:00, 24.0MiB/s]
data.document.fsarrayfile: 100%|██████████| 234k/234k [00:00<00:00, 30.0MiB/s]
data.inverted.bf: 100%|██████████| 362k/362k [00:00<00:00, 19.5MiB/s]
data.lexicon.fsomapfile: 100%|██████████| 682k/682k [00:00<00:00, 1.20MiB/s]
data.lexicon.fsomaphash: 100%|██████████| 777/777 [00:00<00:00, 774kiB/s]
data.lexicon.fsomapid: 100%|██████████| 30.3k/30.3k [00:00<00:00, 15.5MiB/s]
data.meta-0.fsomapfile: 100%|██████████| 725k/725k [00:01<00:00, 684kiB/s] 
data.meta.idx: 100%|██████████| 89.3k/89.3k [00:00<00:00, 18.3MiB/s]
data.meta.zdata: 100%|██████████| 224k/224k [00:00<00:00, 21.8MiB/s]
data.properties: 100%|██████████| 4.29k/4.29k [00:00<?, ?iB/s]
md5sums: 100%|██████████| 619/619 [00:00<00:00, 618kiB/s]


In [21]:
len(retrieval_df['docno'].unique())

11304

In [12]:
retrieval_df.head()

,qid,docid,docno,rank,score,query
0,1,8171,8172,0,24.566031,measurement of dielectric constant of liquids ...
1,1,9880,9881,1,22.110514,measurement of dielectric constant of liquids ...
2,1,5501,5502,2,21.717148,measurement of dielectric constant of liquids ...
3,1,1501,1502,3,19.478355,measurement of dielectric constant of liquids ...
4,1,9858,9859,4,18.626342,measurement of dielectric constant of liquids ...


In [8]:
qrels.head(1)

,qid,docno,label
0,1,1239,1


In [ ]:
train_df = retrieval_df.merge(qrels, how="left", on=["qid", "docno"])
train_df["label"] = train_df["label"].fillna(0).astype(int)
print("samples", len(train_df)) 

samples 91930
